# Differential Equations — Session 31
## Section 7.2: Inverse Transforms and Transforms of Derivatives

**Planned length:** 90 minutes  
**Notebook type:** Student interactive lecture

### Learning objectives

Students should be able to identify inverse transforms, use linearity and partial fractions, transform derivatives, solve constant-coefficient IVPs algebraically in the transform domain, and verify the result against direct or numerical solutions.

**Edition:** Local Instructor Interactive Edition

> **Student interactive edition — local Jupyter/Cursor workflow**
>
> 1. Run the **Local notebook setup** cell below first.
> 2. Read each explanation and derivation in order.
> 3. Run simulation and visualization cells as you reach them.
> 4. At each **Classroom Checkpoint**, stop and work out your answer before class discussion continues.
> 5. This student edition intentionally contains **no instructor answer-reveal cells and no instructor solution notes**.
>
> **Tip:** During class, use `Shift + Enter` to move through the notebook one cell at a time.

In [ ]:
# Local notebook setup — run this cell first.
import importlib.util
import platform
import sys

_REQUIRED = ["numpy", "matplotlib", "scipy", "sympy", "ipywidgets"]
_missing = [name for name in _REQUIRED if importlib.util.find_spec(name) is None]

print(f"Python {sys.version.split()[0]} on {platform.system()}")
if _missing:
    print("Missing packages:", ", ".join(_missing))
    print("From the project folder, run:")
    print("python -m pip install -r requirements.txt")
else:
    print("Student notebook environment is ready.")

### Core 90-minute path

| Time | Topic |
|---:|---|
| 0–18 min | Inverse-transform table |
| 18–38 min | Algebra and partial fractions |
| 38–58 min | Transform of derivatives |
| 58–80 min | Solving an IVP |
| 80–90 min | Verification and exit check |

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import sympy as sp
from scipy.integrate import quad, solve_ivp
from scipy.signal import fftconvolve
from scipy.linalg import expm
from IPython.display import display, Markdown

try:
    from ipywidgets import interact, FloatSlider, IntSlider, Dropdown
    WIDGETS_AVAILABLE = True
except ImportError:
    WIDGETS_AVAILABLE = False

plt.rcParams["figure.figsize"] = (8, 5)
plt.rcParams["axes.grid"] = True
np.set_printoptions(precision=6, suppress=True)

def unit_step(t, a=0.0):
    t = np.asarray(t)
    return (t >= a).astype(float)

print("Notebook ready.")
print("Interactive widgets available:", WIDGETS_AVAILABLE)

## Formal theory reference

### Definition 7.2-A — Inverse Laplace transform

If $\mathcal L\{f\}=F$, then

$$
f(t)=\mathcal L^{-1}\{F(s)\}.
$$

### Theorem 7.2-B — Linearity of the inverse transform

$$
\mathcal L^{-1}\{\alpha F+\beta G\}
=
\alpha f+\beta g.
$$

### Theorem 7.2-C — Transform of a derivative

If the required regularity and growth conditions hold, then

$$
\mathcal L\{f'\}=sF(s)-f(0),
$$

$$
\mathcal L\{f''\}=s^2F(s)-sf(0)-f'(0),
$$

and in general

$$
\mathcal L\{f^{(n)}\}
=
s^nF(s)-s^{n-1}f(0)-\cdots-f^{(n-1)}(0).
$$

### Principle 7.2-D — Laplace solution of an IVP

A constant-coefficient IVP becomes an algebraic equation for $Y(s)$. Solve for $Y$, decompose it into recognizable transforms, and invert.

### Classroom Checkpoint — Derivative Transform

Write $\mathcal L\{y''\}$ in terms of $Y(s)$ and the initial data.

> Pause here. Before continuing, try to explain their reasoning before continuing.

## 1. Inverse transforms by recognition

Examples:

$$
\mathcal L^{-1}\left\{\frac{24}{s^5}\right\}=t^4,
$$

$$
\mathcal L^{-1}\left\{\frac{5}{s^2+25}\right\}=\sin 5t.
$$

In [ ]:
s, t = sp.symbols("s t", positive=True)
examples = [
    24/s**5,
    5/(s**2+25),
    s/(s**2+9),
]
for F in examples:
    display(sp.inverse_laplace_transform(F, s, t))

## 2. Partial fractions

For rational transforms, decompose first.

In [ ]:
F = (2*s+5)/((s-1)*(s-2)*(s+4))
display(sp.apart(F, s))
display(sp.inverse_laplace_transform(F, s, t))

In [ ]:
def pole_response(a=-1.0, b=-3.0):
    t_grid = np.linspace(0, 8, 600)
    response = (np.exp(a*t_grid)-np.exp(b*t_grid))/(a-b)
    plt.plot(t_grid, response)
    plt.xlabel("t")
    plt.ylabel("inverse transform")
    plt.title(fr"$\mathcal{{L}}^{{-1}}\{{1/[(s-{a})(s-{b})]\}}$")
    plt.show()

if WIDGETS_AVAILABLE:
    interact(
        pole_response,
        a=FloatSlider(min=-5, max=2, step=0.25, value=-1),
        b=FloatSlider(min=-6, max=1, step=0.25, value=-3)
    )
else:
    pole_response()

## 3. Why initial conditions appear automatically

Integration by parts gives

$$
\mathcal L\{y'\}
=
sY(s)-y(0).
$$

A second integration gives the $y'(0)$ term for $y''$.

In [ ]:
# Numerical verification for y(t)=e^{-2t}cos(3t)
def y(t):
    return np.exp(-2*t)*np.cos(3*t)
def yp(t):
    return np.exp(-2*t)*(-2*np.cos(3*t)-3*np.sin(3*t))

for s_value in [1, 2, 4]:
    lhs = quad(lambda tau: np.exp(-s_value*tau)*yp(tau), 0, np.inf)[0]
    Y = quad(lambda tau: np.exp(-s_value*tau)*y(tau), 0, np.inf)[0]
    rhs = s_value*Y-y(0)
    print(s_value, lhs, rhs, abs(lhs-rhs))

## 4. Solve an IVP

Solve

$$
y''+4y=6\cos t,
\qquad
y(0)=1,
\qquad
y'(0)=0.
$$

Transforming gives

$$
(s^2Y-s)+4Y=\frac{6s}{s^2+1}.
$$

In [ ]:
Y = sp.solve(
    sp.Eq((s**2+4)*sp.Symbol("Y")-s, 6*s/(s**2+1)),
    sp.Symbol("Y")
)[0]
display(sp.apart(Y, s))
y_exact = sp.simplify(sp.inverse_laplace_transform(Y, s, t))
display(y_exact)

In [ ]:
y_fun = sp.lambdify(t, y_exact, "numpy")
grid = np.linspace(0, 15, 700)

def rhs(t, z):
    return [z[1], 6*np.cos(t)-4*z[0]]

num = solve_ivp(rhs, (0, 15), [1, 0], t_eval=grid, rtol=1e-10, atol=1e-12)

plt.plot(grid, y_fun(grid), label="Laplace solution")
plt.plot(grid, num.y[0], linestyle="--", label="solve_ivp")
plt.legend()
plt.title("IVP solution verification")
plt.show()

print("maximum grid error:", np.max(np.abs(y_fun(grid)-num.y[0])))

## 5. Transform-domain structure

The denominator of $Y(s)$ contains the characteristic information of the differential equation. Poles determine exponential growth, decay, and oscillation in time.

In [ ]:
def second_order_poles(a=2.0, b=5.0):
    roots = np.roots([1, a, b])
    plt.scatter(roots.real, roots.imag, s=100)
    plt.axhline(0); plt.axvline(0)
    plt.xlim(-6, 3); plt.ylim(-5, 5)
    plt.xlabel("real part")
    plt.ylabel("imaginary part")
    plt.title(fr"Poles of $1/(s^2+{a:.2f}s+{b:.2f})$")
    plt.show()
    print("poles:", roots)

if WIDGETS_AVAILABLE:
    interact(
        second_order_poles,
        a=FloatSlider(min=0, max=10, step=0.25, value=2),
        b=FloatSlider(min=0.5, max=15, step=0.5, value=5)
    )
else:
    second_order_poles()

## Classroom Checkpoint — Exit Check

Find

$$
\mathcal L^{-1}\left\{\frac{3s+7}{s^2+4s+13}\right\}.
$$

> Pause here. Let students commit to an answer before running the next cell.